<a href="https://colab.research.google.com/github/Ivan07cabrera/Holamundo/blob/main/Prediccion_compra_casas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install wandb -qU

Uso de wandb para el registro de experimentos.


In [2]:
import wandb
wandb.login()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cabreratrinidadangelivan (cabreratrinidadangelivan-none) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Carga de datos usados para el entrenamiento: https://www.kaggle.com/datasets/mohankrishnathalla/global-house-purchase-decision-dataset

In [3]:
from google.colab import drive #Para montar su drive y que puedan leer los datos
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

import wandb
from wandb.integration.keras import WandbMetricsLogger

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

Base de datos usada


In [9]:
filename = '/content/drive/MyDrive/global_house_purchase_dataset.csv'

In [5]:
df = pd.read_csv('/content/drive/MyDrive/global_house_purchase_dataset.csv')
df

,property_id,country,city,property_type,furnishing_status,property_size_sqft,price,constructed_year,previous_owners,rooms,...,customer_salary,loan_amount,loan_tenure_years,monthly_expenses,down_payment,emi_to_income_ratio,satisfaction_score,neighbourhood_rating,connectivity_score,decision
0,1,France,Marseille,Farmhouse,Semi-Furnished,991,412935,1989,6,6,...,10745,193949,15,6545,218986,0.16,1,5,6,0
1,2,South Africa,Cape Town,Apartment,Semi-Furnished,1244,224538,1990,4,8,...,16970,181465,20,8605,43073,0.08,9,1,2,0
2,3,South Africa,Johannesburg,Farmhouse,Semi-Furnished,4152,745104,2019,5,2,...,21914,307953,30,2510,437151,0.09,6,8,1,0
3,4,Germany,Frankfurt,Farmhouse,Semi-Furnished,3714,1110959,2008,1,3,...,17980,674720,15,8805,436239,0.33,2,6,6,0
4,5,South Africa,Johannesburg,Townhouse,Fully-Furnished,531,99041,2007,6,3,...,17676,65833,25,8965,33208,0.03,3,3,4,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,199996,Germany,Berlin,Villa,Fully-Furnished,685,203328,1968,1,3,...,78330,104050,15,17670,99278,0.01,8,4,5,1
199996,199997,China,Shenzhen,Townhouse,Unfurnished,3818,1454627,1977,5,7,...,25400,1175297,20,2865,279330,0.34,7,10,9,1
199997,199998,Japan,Kyoto,Villa,Semi-Furnished,3603,1619147,1990,2,4,...,28220,743049,30,5595,876098,0.17,5,3,9,0
199998,199999,South Africa,Johannesburg,Apartment,Unfurnished,1706,306165,2010,0,4,...,12240,150774,15,16300,155391,0.11,6,10,6,0


Limpieza de datos


In [6]:
df = df.dropna()

In [7]:
y = df['decision']
x = df.drop(columns=['decision', 'property_id'])

# Variables categóricas
x = pd.get_dummies(x, drop_first=True).astype(float)
x

,property_size_sqft,price,constructed_year,previous_owners,rooms,bathrooms,garage,garden,crime_cases_reported,legal_cases_on_property,...,city_Tokyo,city_Toronto,city_Vancouver,property_type_Farmhouse,property_type_Independent House,property_type_Studio,property_type_Townhouse,property_type_Villa,furnishing_status_Semi-Furnished,furnishing_status_Unfurnished
0,991.0,412935.0,1989.0,6.0,6.0,2.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
1,1244.0,224538.0,1990.0,4.0,8.0,8.0,1.0,1.0,1.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0
2,4152.0,745104.0,2019.0,5.0,2.0,1.0,1.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
3,3714.0,1110959.0,2008.0,1.0,3.0,3.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
4,531.0,99041.0,2007.0,6.0,3.0,3.0,1.0,1.0,3.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
199995,685.0,203328.0,1968.0,1.0,3.0,2.0,0.0,0.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
199996,3818.0,1454627.0,1977.0,5.0,7.0,5.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0
199997,3603.0,1619147.0,1990.0,2.0,4.0,4.0,1.0,1.0,1.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
199998,1706.0,306165.0,2010.0,0.0,4.0,1.0,1.0,0.0,0.0,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


Preparación de datos de entrenamiento, prueba y validación

In [8]:
x_train, x_temp, y_train, y_temp = train_test_split(
    x, y, test_size=0.3, random_state=42
)

x_test, x_val, y_test, y_val = train_test_split(
    x_temp, y_temp, test_size=1/3, random_state=42
)

In [9]:
scaler = StandardScaler()

x_train = scaler.fit_transform(x_train)
x_test = scaler.transform(x_test)
x_val = scaler.transform(x_val)

Funciones a usar para la creación de redes con diferentes arquitecturas.


In [10]:
def train_experiment(config=None):

    with wandb.init(project="prediccion de compra de casas1", config=config):
        config = wandb.config

        model = Sequential()

        # Capas de red
        for units in config.hidden_layers:
            model.add(Dense(units, activation='relu'))

            if config.use_dropout:
                model.add(Dropout(config.dropout_rate))

        # Ultima capa
        model.add(Dense(1, activation='sigmoid'))


        model.compile(
            optimizer=config.optimizer,
            loss='binary_crossentropy',
            metrics=['accuracy']
        )

        history = model.fit(
            x_train, y_train,          #Entrenamiento
            epochs=config.epochs,
            batch_size=config.batch_size,
            validation_data=(x_test, y_test),
            callbacks=[WandbMetricsLogger()],
            verbose=1
        )

        # Evaluación
        test_loss, test_acc = model.evaluate(x_test, y_test, verbose=1)

        wandb.log({
            "test_accuracy_final": test_acc,
            "test_loss_final": test_loss
        })

        return model, history



Arquitecturas de redes escogidas.

In [11]:
experiments = [
    {
        "hidden_layers": [32],
        "optimizer": "adam",
        "epochs": 20,
        "batch_size": 32,
        "use_dropout": False,
        "dropout_rate": 0.0
    },
    {
        "hidden_layers": [64, 32],
        "optimizer": "adam",
        "epochs": 20,
        "batch_size": 32,
        "use_dropout": True,
        "dropout_rate": 0.3
    },
    {
        "hidden_layers": [128, 64, 32],
        "optimizer": "adam",
        "epochs": 20,
        "batch_size": 32,
        "use_dropout": True,
        "dropout_rate": 0.4
    },
    {
        "hidden_layers": [ 32],
        "optimizer": "rmsprop",
        "epochs": 20,
        "batch_size": 32,
        "use_dropout": False,
        "dropout_rate": 0.0
    },
    {
        "hidden_layers": [64, 32],
        "optimizer": "rmsprop",
        "epochs": 20,
        "batch_size": 32,
        "use_dropout": True,
        "dropout_rate": 0.3
    },
    {
        "hidden_layers": [128, 64, 32],
        "optimizer": "rmsprop",
        "epochs": 20,
        "batch_size": 32,
        "use_dropout": True,
        "dropout_rate": 0.4
    }

]

Inicio de entrenamiento:

In [12]:
models = []
histories = []
test_accuracies = []

for exp in experiments:
    model, history = train_experiment(exp)

    models.append(model)
    histories.append(history)

    acc = history.history['val_accuracy'][-1]
    test_accuracies.append(acc)


Epoch 1/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9484 - loss: 0.1282 - val_accuracy: 0.9749 - val_loss: 0.0662
Epoch 2/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9887 - loss: 0.0368 - val_accuracy: 0.9948 - val_loss: 0.0185
Epoch 3/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9967 - loss: 0.0113 - val_accuracy: 0.9972 - val_loss: 0.0081
Epoch 4/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9978 - loss: 0.0063 - val_accuracy: 0.9972 - val_loss: 0.0066
Epoch 5/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9982 - loss: 0.0048 - val_accuracy: 0.9976 - val_loss: 0.0056
Epoch 6/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 10s 1ms/step - accuracy: 0.9982 - loss: 0.0043 - val_accuracy: 0.9979 - val_loss: 0.0051
Epoch 7/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9985 - loss: 0.0038 - val_accuracy: 0.9978 - val_loss: 0.0046
Epoch 8/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9986 - loss: 0.0034 -

epoch/accuracy,▁▇██████████████████
epoch/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▇██████████████████
epoch/val_loss,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy_final,▁
test_loss_final,▁
epoch/accuracy,0.99921
epoch/epoch,19
epoch/learning_rate,0.001


Epoch 1/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9411 - loss: 0.1344 - val_accuracy: 0.9918 - val_loss: 0.0228
Epoch 2/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9930 - loss: 0.0195 - val_accuracy: 0.9971 - val_loss: 0.0070
Epoch 3/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9961 - loss: 0.0108 - val_accuracy: 0.9974 - val_loss: 0.0062
Epoch 4/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9965 - loss: 0.0092 - val_accuracy: 0.9975 - val_loss: 0.0051
Epoch 5/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9968 - loss: 0.0086 - val_accuracy: 0.9973 - val_loss: 0.0076
Epoch 6/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9971 - loss: 0.0077 - val_accuracy: 0.9981 - val_loss: 0.0051
Epoch 7/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 6s 1ms/step - accuracy: 0.9976 - loss: 0.0061 - val_accuracy: 0.9984 - val_loss: 0.0039
Epoch 8/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9974 - loss: 0.0070 - 

epoch/accuracy,▁▇██████████████████
epoch/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▆▇▇▇▇█████▇████████
epoch/val_loss,█▂▂▂▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy_final,▁
test_loss_final,▁
epoch/accuracy,0.99842
epoch/epoch,19
epoch/learning_rate,0.001


Epoch 1/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 10s 2ms/step - accuracy: 0.9398 - loss: 0.1386 - val_accuracy: 0.9926 - val_loss: 0.0207
Epoch 2/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9908 - loss: 0.0244 - val_accuracy: 0.9978 - val_loss: 0.0065
Epoch 3/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9951 - loss: 0.0135 - val_accuracy: 0.9972 - val_loss: 0.0064
Epoch 4/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9961 - loss: 0.0110 - val_accuracy: 0.9976 - val_loss: 0.0051
Epoch 5/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9963 - loss: 0.0100 - val_accuracy: 0.9975 - val_loss: 0.0058
Epoch 6/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9965 - loss: 0.0091 - val_accuracy: 0.9982 - val_loss: 0.0047
Epoch 7/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9969 - loss: 0.0080 - val_accuracy: 0.9981 - val_loss: 0.0045
Epoch 8/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9971 - loss: 0.0078 -

...

epoch/accuracy,▁▇██████████████████
epoch/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▃▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▆▇▇▇███████████████
epoch/val_loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
test_accuracy_final,▁
test_loss_final,▁
epoch/accuracy,0.99928
epoch/epoch,19
epoch/learning_rate,0.001


Epoch 1/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9370 - loss: 0.1430 - val_accuracy: 0.9860 - val_loss: 0.0360
Epoch 2/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9896 - loss: 0.0285 - val_accuracy: 0.9972 - val_loss: 0.0073
Epoch 3/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9953 - loss: 0.0134 - val_accuracy: 0.9974 - val_loss: 0.0062
Epoch 4/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9962 - loss: 0.0113 - val_accuracy: 0.9984 - val_loss: 0.0042
Epoch 5/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9969 - loss: 0.0102 - val_accuracy: 0.9980 - val_loss: 0.0045
Epoch 6/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9971 - loss: 0.0096 - val_accuracy: 0.9977 - val_loss: 0.0063
Epoch 7/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 1ms/step - accuracy: 0.9973 - loss: 0.0092 - val_accuracy: 0.9979 - val_loss: 0.0057
Epoch 8/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9973 - loss: 0.0094 - 

epoch/accuracy,▁▇██████████████████
epoch/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▇▇██▇▇██▇██████████
epoch/val_loss,█▂▂▁▁▂▁▁▁▂▁▁▁▁▁▁▁▁▁▁
test_accuracy_final,▁
test_loss_final,▁
epoch/accuracy,0.99807
epoch/epoch,19
epoch/learning_rate,0.001


Epoch 1/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.9380 - loss: 0.1485 - val_accuracy: 0.9851 - val_loss: 0.0392
Epoch 2/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.9867 - loss: 0.0399 - val_accuracy: 0.9961 - val_loss: 0.0111
Epoch 3/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9932 - loss: 0.0236 - val_accuracy: 0.9970 - val_loss: 0.0084
Epoch 4/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9945 - loss: 0.0198 - val_accuracy: 0.9966 - val_loss: 0.0094
Epoch 5/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 9s 2ms/step - accuracy: 0.9956 - loss: 0.0173 - val_accuracy: 0.9974 - val_loss: 0.0064
Epoch 6/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 7s 2ms/step - accuracy: 0.9962 - loss: 0.0164 - val_accuracy: 0.9974 - val_loss: 0.0073
Epoch 7/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9965 - loss: 0.0155 - val_accuracy: 0.9978 - val_loss: 0.0053
Epoch 8/20
4375/4375 ━━━━━━━━━━━━━━━━━━━━ 8s 2ms/step - accuracy: 0.9968 - loss: 0.0141 - 

epoch/accuracy,▁▇▇█████████████████
epoch/epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
epoch/learning_rate,▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
epoch/val_accuracy,▁▇▇▇████████████████
epoch/val_loss,█▂▂▂▁▁▁▁▁▂▁▂▂▂▂▂▂▂▃▂
test_accuracy_final,▁
test_loss_final,▁
epoch/accuracy,0.99781
epoch/epoch,19
epoch/learning_rate,0.001


Obtención del "Mejor arquitectura " y validación.

In [13]:
best_index = np.argmax(test_accuracies)
best_model = models[best_index]

print(f"Mejor modelo: Experimento {best_index}")
print(f"Test accuracy: {test_accuracies[best_index]}")

Mejor modelo: Experimento 2
Test accuracy: 0.9988750219345093


In [14]:
chosen_index = 1

chosen_model = models[chosen_index]

print(f"Modelo elegido: Experimento {chosen_index}")

print(
    f"Test accuracy: "
    f"{test_accuracies[chosen_index]}"
)

Modelo elegido: Experimento 1
Test accuracy: 0.9984999895095825


In [17]:
y_pred = (
    chosen_model.predict(x_val) > 0.5
).astype(int)

acc_val = accuracy_score(
    y_val,
    y_pred
)

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step


In [18]:
print(acc_val)

0.99825


In [20]:
chosen_model.save("modelo_completo.keras")

chosen_model.save_weights(
    "pesos_modelo.weights.h5"
)

In [41]:
a=1
print(a)

1
